# **Etapa 02 - Modelagem com Redes Neurais**

**Objetivo:** Construir MLP utilizando PyTorch e comparar métricas do mesmo com modelos lineares e de árvores.

## **Imports e Setup Inicial de Valores**

In [126]:
#!pip install mlflow

In [127]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.compose import ColumnTransformer
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
import logging
import mlflow

RANDOM_STATE = 17
TEST_SIZE = 0.2

# Declaração do logger no escopo global
logger = logging.getLogger("tc_etapa_02")


## **Método para configuração do Logger**

In [128]:
def configurar_logging(nivel=logging.INFO):
    """
    Configura o logger global. Pode ser chamado múltiplas vezes
    para resetar as configurações durante a sessão.
    """
    # Limpa handlers existentes para evitar duplicação de logs
    if logger.hasHandlers():
        logger.handlers.clear()

    logger.setLevel(nivel)
    logger.propagate = False

    # Define o formato
    formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')

    # Handler para o console (saída no notebook)
    stream_handler = logging.StreamHandler()
    stream_handler.setFormatter(formatter)
    logger.addHandler(stream_handler)


## **Carregamento de Dados**

In [129]:
def carregar_dados(path):
    """
    Carrega um arquivo Excel e retorna um DataFrame.
    """
    logger.info(f"Carregando dados do arquivo: {path}")
    df = pd.read_excel(path)
    return df

## **Tratamento dos dados**

In [130]:
def padronizar_nomes_features(df):
    """
    Padroniza os nomes das colunas para lowercase e substitui espaços por underscores.
    """
    logger.info("Padronizando nomes das colunas")
    df.columns = [c.lower().replace(" ", "_") for c in df.columns]
    return df

In [131]:
def corrigir_feature_total_charges(df):
  """
  Corrige a feature 'total_charges' para float64 e substitui NaN por 0.
  """
  logger.info("Corrigindo feature 'total_charges'")
  df['total_charges'] = pd.to_numeric(df['total_charges'], errors='coerce')
  df.fillna({'total_charges': 0}, inplace=True)
  return df

In [132]:
def remover_features_irrelevantes(df):
  """
  Remove features irrelevantes para o modelo.
  """
  logger.info("Removendo features irrelevantes")

  cols_to_drop = [
    "customerid",
    "count",
    "country",
    "state",
    "city",
    "lat_long",
    "latitude",
    "longitude",
    "churn_label",
    "churn_score",
    "cltv",
    "churn_reason",
    "total_charges",
    "tenure_months"
  ]
  logger.debug(f"Features a serem removidas: {cols_to_drop}")
  df = df.drop(columns=cols_to_drop)
  return df

In [133]:
def criar_feature_average_monthly_spend(df):
  """
  Cria a feature 'average_monthly_spend' a partir de 'total_charges' e 'tenure_months'.
  """
  logger.info("Criando feature 'average_monthly_spend'")
  df['average_monthly_spend'] = df['total_charges'] / df['tenure_months']
  df['average_monthly_spend'] = df['average_monthly_spend'].replace([np.inf, -np.inf], 0).fillna(0)
  return df

In [134]:
def aplicar_feature_engineering(df):
  """
  Aplica todas as transformações de feature engineering.
  """
  logger.info("Aplicando feature engineering")
  df = criar_feature_average_monthly_spend(df)
  return df

In [135]:
def tratar_dados(df):
    """
    Aplica todos os tratamentos de dados.
    """
    logger.info("Tratando dados")
    df = padronizar_nomes_features(df)
    df = corrigir_feature_total_charges(df)
    df = aplicar_feature_engineering(df)
    df = remover_features_irrelevantes(df)
    return df

## **Separação de Dados de Treino e Teste**

In [136]:
def separar_dados_treino_teste(df):
  """
  Separa os dados em treino e teste.
  """
  logger.info("Separando dados em treino e teste")

  X = df.drop('churn_value', axis=1)
  y = df['churn_value']

  X_train, X_test, y_train, y_test = train_test_split(
      X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
  )

  return X_train, X_test, y_train, y_test

## **Construir Transformer para One-Hot Encoding**

In [137]:
def construir_transformer_hot_encoding(X_train):
  """
  Cria um transformer para one-hot encoding.
  """
  logger.info("Construindo transformer para one-hot encoding")

  colunas_numericas = X_train.select_dtypes(include=['int64', 'float64']).columns
  colunas_categoricas = X_train.select_dtypes(include=['object']).columns

  transformer = ColumnTransformer(
      transformers=[
          ('numerics', StandardScaler(), colunas_numericas),
          ('categoricals', OneHotEncoder(drop='first', handle_unknown='ignore'), colunas_categoricas)
      ]
  )

  return transformer

## **Construir Balanceador**

In [138]:
def construir_balanceador():
  """
  Cria um balanceador SMOTE.
  """
  logger.info("Construindo balanceador SMOTE")

  return SMOTE(random_state=RANDOM_STATE)

## **Construir Pipeline**

In [139]:
def construir_pipeline(nome_execucao, transformer, balanceador, modelo):
  """
  Cria um pipeline com o transformer, balanceador e modelo.
  """
  logger.info(f"Construindo pipeline para '{nome_execucao}'")

  pipeline = Pipeline(
    steps=[
        ('pre-processamento', transformer),
        ('balanceamento', balanceador),
        ('classificador', modelo)
    ]
  )

  return pipeline

## **Executar validação cruzada**

In [140]:
def executar_validacao_cruzada(nome_execucao, pipeline, X_train, y_train, folds=5):
  """
  Executa a validação cruzada com o pipeline e retorna os resultados.
  """
  logger.info(f"Executando validação cruzada para '{nome_execucao}'")

  metricas = ['accuracy', 'precision', 'recall', 'f1']
  resultados_cv = cross_validate(
      pipeline, X_train, y_train, cv=5, scoring=metricas
  )

  logger.debug(f"--- FASE DE VALIDAÇÃO CRUZADA (Média das {folds} Pastas) ---")
  logger.debug(f"Acurácia Média:   {resultados_cv['test_accuracy'].mean():.4f}")
  logger.debug(f"Precisão Média: {resultados_cv['test_precision'].mean():.4f}")
  logger.debug(f"Recall Médio:    {resultados_cv['test_recall'].mean():.4f}")
  logger.debug(f"F1-Score Médio:  {resultados_cv['test_f1'].mean():.4f}\n")

## **Treinamento e Teste do Modelo**

In [141]:
def treinar_e_testar_modelo(nome_execucao, pipeline, X_train, X_test, y_train, y_test):
  """
  Treina e testa o modelo.
  """
  logger.info(f"Treinando e testando modelo '{nome_execucao}'")

  pipeline.fit(X_train, y_train)

  previsoes = {}

  previsoes_train = pipeline.predict(X_train)
  previsoes_test = pipeline.predict(X_test)

  previsoes["previsoes_train"] = previsoes_train
  previsoes["previsoes_test"] = previsoes_test

  if (hasattr(pipeline, "predict_proba")):
    previsoes_train_proba = pipeline.predict_proba(X_train)[:, 1]
    previsoes_test_proba = pipeline.predict_proba(X_test)[:, 1]
  elif (hasattr(pipeline, "decision_function")):
    previsoes_train_proba = pipeline.decision_function(X_train)
    previsoes_test_proba = pipeline.decision_function(X_test)

  previsoes["previsoes_train_proba"] = previsoes_train_proba
  previsoes["previsoes_test_proba"] = previsoes_test_proba

  return previsoes

## **Calcular Métricas**

In [142]:
def calcular_metricas(y_target, previsoes, label):
  """
  Calcula as métricas de avaliação.
  """
  logger.info("Calculando métricas")

  acuracia = accuracy_score(y_target, previsoes)
  precisao = precision_score(y_target, previsoes, pos_label=1)
  recall = recall_score(y_target, previsoes, pos_label=1)
  f1 = f1_score(y_target, previsoes, pos_label=1)

  logger.debug("--- MÉTRICAS DE EXECUÇÃO DO MODELO ---")
  logger.debug(f"Acurácia no {label}:  {acuracia:.4f}")
  logger.debug(f"Precisão no {label}: {precisao:.4f}")
  logger.debug(f"Recall no {label}:    {recall:.4f}")
  logger.debug(f"F1-Score no {label}:  {f1:.4f}\n")

  logger.debug("--- RELATÓRIO DE CLASSIFICAÇÃO DETALHADO ---")
  report = classification_report(y_target, previsoes)
  logger.debug(f"\n{report}")

  return acuracia, precisao, recall, f1



## **Calcular Métricas de Treino e Teste**

In [143]:
def calcular_metricas_treino_teste(nome_execucao, y_train, y_test, previsoes):
  """
  Calcula as métricas de treino e teste.
  """
  logger.info(f"Calculando métricas de treino e teste para '{nome_execucao}")

  train_accuracy, train_precision, train_recall, train_f1 = calcular_metricas(y_train, previsoes["previsoes_train"], "TREINO")
  test_accuracy, test_precision, test_recall, test_f1 = calcular_metricas(y_test, previsoes["previsoes_test"], "TESTE")

  overfitting = train_accuracy - test_accuracy

  logger.debug("--- OVERFITTING ---")
  logger.debug(f"Overfitting: {overfitting:.4f}")

  metricas = {}
  metricas["train_accuracy"] = train_accuracy
  metricas["test_accuracy"] = test_accuracy
  metricas["train_precision"] = train_precision
  metricas["test_precision"] = test_precision
  metricas["train_recall"] = train_recall
  metricas["test_recall"] = test_recall
  metricas["train_f1"] = train_f1
  metricas["test_f1"] = test_f1
  metricas["overfitting"] = overfitting

  return metricas

## **Registrar Execução no MLFlow**

In [144]:
def registrar_execução_mlflow(
    nome_execucao,
    metricas,
    model
):
  """
  Registra a execução no MLFlow.
  """
  logger.info("Registrando execução no MLFlow")
  with mlflow.start_run(run_name=nome_execucao):
    mlflow.log_metric("train_accuracy", metricas["train_accuracy"])
    mlflow.log_metric("test_accuracy", metricas["test_accuracy"])
    mlflow.log_metric("train_precision", metricas["train_precision"])
    mlflow.log_metric("test_precision", metricas["test_precision"])
    mlflow.log_metric("train_recall", metricas["train_recall"])
    mlflow.log_metric("test_recall", metricas["test_recall"])
    mlflow.log_metric("train_f1_score", metricas["train_f1"])
    mlflow.log_metric("test_f1_score", metricas["test_f1"])
    mlflow.log_metric("overfitting", metricas["overfitting"])
    mlflow.sklearn.log_model(model, "model")

## **Avaliar Modelo**

In [145]:
def avaliar_modelo(nome_execucao, transformer, balanceador, modelo, X_train, y_train, X_test, y_test):
  """
  Avalia o modelo.
  """
  logger.info("Avaliando modelo")

  pipeline = construir_pipeline(nome_execucao, transformer, balanceador, modelo)

  executar_validacao_cruzada(nome_execucao, pipeline, X_train, y_train)

  previsoes = treinar_e_testar_modelo(nome_execucao, pipeline, X_train, X_test, y_train, y_test)

  metricas = calcular_metricas_treino_teste(nome_execucao, y_train, y_test, previsoes)

  registrar_execução_mlflow(
      nome_execucao,
      metricas,
      modelo
  )

## **Configurar MLFlow**

In [146]:
def configurar_mlflow():
  """
  Configura o MLFlow.
  """
  logger.info("Configurando MLFlow")
  mlflow.set_tracking_uri("/content/mlflow")
  mlflow.set_experiment("TechChallenge - Etapa 02")


## **Executar Modelo LogisticRegression**

In [147]:
def executar_logistic_regression(X_train, X_test, y_train, y_test, transformer, balanceador):
  """
  Executa o modelo LogisticRegression.
  """
  logger.info("Executando modelo LogisticRegression")

    # Obter modelo
  modelo = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000, class_weight="balanced")

  # Avaliar modelo
  avaliar_modelo("Logistic Regression", transformer, balanceador, modelo, X_train, y_train, X_test, y_test)

## **Executar Modelo DecisionTreeClassifier**

In [148]:
def executar_decision_tree_classifier(X_train, X_test, y_train, y_test, transformer, balanceador):
  """
  Executa o modelo DecisionTreeClassifier.
  """
  logger.info("Executando modelo DecisionTreeClassifier")

  # Obter modelo
  modelo = DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight="balanced", max_depth=5)

  # Avaliar modelo
  avaliar_modelo("Decision Tree Classifier", transformer, balanceador, modelo, X_train, y_train, X_test, y_test)

## **Executar Modelo RandomForestClassifier**

In [149]:
def executar_random_forest_classifier(X_train, X_test, y_train, y_test, transformer, balanceador):
  """
  Executa o modelo RandomForestClassifier.
  """
  logger.info("Executando modelo RandomForestClassifier")

  # Obter modelo
  modelo = RandomForestClassifier(random_state=RANDOM_STATE, max_depth=5)

  # Avaliar modelo
  avaliar_modelo("Random Forest Classifier", transformer, balanceador, modelo, X_train, y_train, X_test, y_test)

## **Executar Modelo GradientBoostingClassifier**

In [150]:
def executar_gradient_boosting_classifier(X_train, X_test, y_train, y_test, transformer, balanceador):
  """
  Executa o modelo GradientBoostingClassifier.
  """
  logger.info("Executando modelo GradientBoostingClassifier")

  # Obter modelo
  modelo = GradientBoostingClassifier(random_state=RANDOM_STATE, max_depth=5)

  # Avaliar modelo
  avaliar_modelo("Gradient Boosting Classifier", transformer, balanceador, modelo, X_train, y_train, X_test, y_test)

## **Execução Geral do 'Programa'**

In [151]:
def main():
  logger.info("Iniciando o programa")

  # Configurar o Logger
  configurar_logging(logging.DEBUG)

  # Carregar dados
  df = carregar_dados("/content/Telco_customer_churn.xlsx")

  # Tratamento dos dados
  df = tratar_dados(df)

  # Separar dados de treino e teste
  X_train, X_test, y_train, y_test = separar_dados_treino_teste(df)

  # Obter transformer
  transformer = construir_transformer_hot_encoding(X_train)

  # Obter balanceador
  balanceador = construir_balanceador()

  # Configurar o MLFlow
  configurar_mlflow()

  # Executar o modelo LogisticRegression
  executar_logistic_regression(X_train, X_test, y_train, y_test, transformer, balanceador)

  # Executar o modelo DecisionTreeClassifier
  executar_decision_tree_classifier(X_train, X_test, y_train, y_test, transformer, balanceador)

  # Executar o modelo RandomForestClassifier
  executar_random_forest_classifier(X_train, X_test, y_train, y_test, transformer, balanceador)

  # Executar o modelo GradientBoostingClassifier
  executar_gradient_boosting_classifier(X_train, X_test, y_train, y_test, transformer, balanceador)

  logger.info("Programa finalizado")

In [152]:
main()

2026-05-29 20:09:08,331 - tc_etapa_02 - INFO - Iniciando o programa
2026-05-29 20:09:08,335 - tc_etapa_02 - INFO - Carregando dados do arquivo: /content/Telco_customer_churn.xlsx
2026-05-29 20:09:12,938 - tc_etapa_02 - INFO - Tratando dados
2026-05-29 20:09:12,939 - tc_etapa_02 - INFO - Padronizando nomes das colunas
2026-05-29 20:09:12,941 - tc_etapa_02 - INFO - Corrigindo feature 'total_charges'
2026-05-29 20:09:12,946 - tc_etapa_02 - INFO - Aplicando feature engineering
2026-05-29 20:09:12,946 - tc_etapa_02 - INFO - Criando feature 'average_monthly_spend'
2026-05-29 20:09:12,950 - tc_etapa_02 - INFO - Removendo features irrelevantes
2026-05-29 20:09:12,950 - tc_etapa_02 - DEBUG - Features a serem removidas: ['customerid', 'count', 'country', 'state', 'city', 'lat_long', 'latitude', 'longitude', 'churn_label', 'churn_score', 'cltv', 'churn_reason', 'total_charges', 'tenure_months']
2026-05-29 20:09:12,955 - tc_etapa_02 - INFO - Separando dados em treino e teste
2026-05-29 20:09:12,96